In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir('/home/jhpark/image-artifacts/src')
os.environ["CUDA_VISIBLE_DEVICES"] = "2"


In [3]:
# Install required packages if needed
# !pip install torch torchvision transformers openai PIL numpy tqdm matplotlib
# !pip install groundingdino-py segment-anything

import sys
import uuid
import json
import time
import logging
import pickle
import random
import shutil
from datetime import datetime
from typing import List, Dict, Optional, Tuple, Any
import traceback
from collections import defaultdict

import numpy as np
import openai
import torch
from tqdm import tqdm
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import matplotlib.pyplot as plt

# Add pipeline to path

from pipeline import (
    GSAMDetector, InstanceProcessor, ImageVisualizer,
)
from pipeline.data_loader import _initialize_data_loader, _get_image_list
from pipeline.prompts import get_entity_subentities, MoneyManager

/home/jhpark/anaconda3/envs/gsam/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Configuration - modify these as needed
CONFIG = {
    # Dataset configuration
    'dataset_type': 'coco',
    'dataset_path': '/data3/jhpark/coco/annotations',
    'image_path': '/data3/jhpark/coco/train2017',
    'super_categories': ['animal'],  # Process animal images
    
    # Processing parameters
    'max_images': 1,  # Process only one image
    'max_artifacts_per_image': 2,  # Generate 2 artifacts per image
    'artifact_types': ['fusion', 'distortion', 'removal', 'addition'],
    
    # GSAM parameters
    'min_area_ratio': 0.005,
    'max_area_ratio': 0.5,
    'box_threshold': 0.3,
    'text_threshold': 0.25,
    'device': 'cuda:0' if torch.cuda.is_available() else 'cpu',
    
    # Model paths (adjust if needed)
    'grounding_config_file': 'GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py',
    'grounding_checkpoint': 'weight/groundingdino_swint_ogc.pth',
    'sam_version': 'vit_h',
    'sam_checkpoint': 'weight/sam_vit_h_4b8939.pth',
    'sam_hq_checkpoint': None,
    'use_sam_hq': False,
    'bert_base_uncased_path': None,
    
    # FLUX parameters
    'guidance': 5.0,
    'num_steps': 25,
    'inject_step': 20,
    'pe_step_addition': 25,
    'pe_step_removal': 25,
    'pe_step_distortion': 20,
    'pe_step_fusion': 20,
    'seed': 42,
    'use_rf_solver': False
}

# Set random seed for reproducibility
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

Configuration loaded:
  dataset_type: coco
  dataset_path: /data3/jhpark/coco/annotations
  image_path: /data3/jhpark/coco/train2017
  super_categories: ['animal']
  max_images: 1
  max_artifacts_per_image: 2
  artifact_types: ['fusion', 'distortion', 'removal', 'addition']
  min_area_ratio: 0.005
  max_area_ratio: 0.5
  box_threshold: 0.3
  text_threshold: 0.25
  device: cuda:0
  grounding_config_file: GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py
  grounding_checkpoint: weight/groundingdino_swint_ogc.pth
  sam_version: vit_h
  sam_checkpoint: weight/sam_vit_h_4b8939.pth
  sam_hq_checkpoint: None
  use_sam_hq: False
  bert_base_uncased_path: None
  guidance: 5.0
  num_steps: 25
  inject_step: 20
  pe_step_addition: 25
  pe_step_removal: 25
  pe_step_distortion: 20
  pe_step_fusion: 20
  seed: 42
  use_rf_solver: False


In [5]:
# Initialize OpenAI client
if not os.getenv('OPENAI_API_KEY'):
    print("❌ Error: OPENAI_API_KEY environment variable not set.")
    print("Please set your OpenAI API key:")
    print("  export OPENAI_API_KEY='your-api-key-here'")
else:
    openai_client = openai.OpenAI()
    print("✅ OpenAI client initialized")

# Initialize MoneyManager for tracking API costs
money_manager = MoneyManager(model="gpt-4o")

✅ OpenAI client initialized


In [6]:
data_loader = _initialize_data_loader(CONFIG['dataset_type'], CONFIG)


loading annotations into memory...
Done (t=0.78s)
creating index...
index created!
loading annotations into memory...
Done (t=14.38s)
creating index...
index created!


In [7]:
if CONFIG['dataset_type'].lower() == 'coco':
    supercategories = set()
    for cat in data_loader.coco_class.dataset['categories']:
        supercategories.add(cat['supercategory'])
    print("COCO supercategories:", sorted(supercategories))

coco_data = {}
for sup_cat in supercategories:
    image_list = _get_image_list(
        CONFIG['dataset_type'], 
        data_loader, 
        [sup_cat], 
        max_instances_per_image=3,
        logger=logging.getLogger()
    )
    coco_data[sup_cat] = image_list

COCO supercategories: ['accessory', 'animal', 'appliance', 'electronic', 'food', 'furniture', 'indoor', 'kitchen', 'outdoor', 'person', 'sports', 'vehicle']
Counting instances per image...
number of images 6122
Counting instances per image...
number of images 1568
Counting instances per image...
number of images 1348
Counting instances per image...
number of images 2780
Counting instances per image...
number of images 1787
Counting instances per image...
number of images 10447
Counting instances per image...
number of images 3660
Counting instances per image...
number of images 4159
Counting instances per image...
number of images 5691
Counting instances per image...
number of images 957
Counting instances per image...
number of images 9814
Counting instances per image...
number of images 2523


In [8]:
for key, value in coco_data.items():
    print(key, len(value))

sports 6122
accessory 1568
electronic 1348
outdoor 2780
appliance 1787
person 10447
indoor 3660
furniture 4159
vehicle 5691
kitchen 957
animal 9814
food 2523


In [9]:
def regroup_supercategories(original_data, merge_groups):
    """
    Regroup supercategories by merging specified categories.
    
    Args:
        original_data (dict): Original dictionary with supercategories as keys
        merge_groups (dict): Dictionary where keys are new group names and values are lists of supercategories to merge
                            Example: {'objects': ['furniture', 'appliance'], 'living': ['person', 'animal']}
    
    Returns:
        dict: New dictionary with regrouped supercategories
    """
    regrouped_data = {}
    used_categories = set()
    
    # Process merge groups
    for new_group_name, categories_to_merge in merge_groups.items():
        merged_images = []
        print(f"\nMerging {categories_to_merge} into '{new_group_name}':")
        
        for category in categories_to_merge:
            if category in original_data:
                images = original_data[category]
                merged_images.extend(images)
                used_categories.add(category)
                print(f"  - {category}: {len(images)} images")
            else:
                print(f"  - Warning: {category} not found in original data")
        
        regrouped_data[new_group_name] = merged_images
        print(f"  Total in '{new_group_name}': {len(merged_images)} images")
    
    # Add remaining categories that weren't merged
    for category, images in original_data.items():
        if category not in used_categories:
            regrouped_data[category] = images
            print(f"\nKeeping '{category}' unchanged: {len(images)} images")
    
    return regrouped_data

# Display current supercategories and their counts
print("Current supercategories:")
for key, value in coco_data.items():
    print(f"  {key}: {len(value)} images")

print(f"\nTotal images: {sum(len(v) for v in coco_data.values())}")
print(f"Total categories: {len(coco_data)}")


Current supercategories:
  sports: 6122 images
  accessory: 1568 images
  electronic: 1348 images
  outdoor: 2780 images
  appliance: 1787 images
  person: 10447 images
  indoor: 3660 images
  furniture: 4159 images
  vehicle: 5691 images
  kitchen: 957 images
  animal: 9814 images
  food: 2523 images

Total images: 50856
Total categories: 12


In [10]:
# Example usage: Define which supercategories you want to merge
# Customize this dictionary according to your needs
merge_groups = {
    'objects': ['furniture', 'kitchen', 'appliance', 'electronic', 'food', 'accessory'],  # Physical objects
    'person': ['person'],               # Living entities  
    'animal': ['animal'],               # Living entities  
    'scene': ['indoor', 'outdoor'],
    'sports_vehicle': ['sports', 'vehicle'],
}

print("="*50)
print("Regrouping supercategories...")
print("="*50)

# Apply the regrouping
regrouped_coco_data = regroup_supercategories(coco_data, merge_groups)

print("\n" + "="*50)
print("Final regrouped data summary:")
print("="*50)
for key, value in regrouped_coco_data.items():
    print(f"  {key}: {len(value)} images")
print(f"\nTotal images: {sum(len(v) for v in regrouped_coco_data.values())}")
print(f"Total categories: {len(regrouped_coco_data)}")


Regrouping supercategories...

Merging ['furniture', 'kitchen', 'appliance', 'electronic', 'food', 'accessory'] into 'objects':
  - furniture: 4159 images
  - kitchen: 957 images
  - appliance: 1787 images
  - electronic: 1348 images
  - food: 2523 images
  - accessory: 1568 images
  Total in 'objects': 12342 images

Merging ['person'] into 'person':
  - person: 10447 images
  Total in 'person': 10447 images

Merging ['animal'] into 'animal':
  - animal: 9814 images
  Total in 'animal': 9814 images

Merging ['indoor', 'outdoor'] into 'scene':
  - indoor: 3660 images
  - outdoor: 2780 images
  Total in 'scene': 6440 images

Merging ['sports', 'vehicle'] into 'sports_vehicle':
  - sports: 6122 images
  - vehicle: 5691 images
  Total in 'sports_vehicle': 11813 images

Final regrouped data summary:
  objects: 12342 images
  person: 10447 images
  animal: 9814 images
  scene: 6440 images
  sports_vehicle: 11813 images

Total images: 50856
Total categories: 5


In [11]:
for key, value in regrouped_coco_data.items():
    print(key, len(value))


objects 12342
person 10447
animal 9814
scene 6440
sports_vehicle 11813


In [12]:
import os
import shutil

# Define the output directory for regrouped images
output_dir = "/data3/jhpark/image-artifact-real-images/coco"
os.makedirs(output_dir, exist_ok=True)

for group_name, image_paths in regrouped_coco_data.items():
    group_dir = os.path.join(output_dir, group_name)
    os.makedirs(group_dir, exist_ok=True)
    for data in image_paths:
        # Get the image filename
        img_filename = os.path.join('/data3/jhpark/coco/train2017', data['file_name'])
        dest_path = os.path.join(group_dir, data['file_name'])
        # Copy the image to the group directory
        shutil.copy2(img_filename, dest_path)


In [13]:
# # Helper function to create custom regroupings
# def create_custom_merge_groups():
#     """
#     Helper function to interactively create merge groups.
#     Modify this section to define your custom groupings.
#     """
    
#     print("Available supercategories to regroup:")
#     categories = list(coco_data.keys())
#     for i, cat in enumerate(categories):
#         print(f"  {i+1:2d}. {cat:<12} ({len(coco_data[cat]):,} images)")
    
#     print("\n" + "="*60)
#     print("Common regrouping strategies:")
#     print("="*60)
    
#     # Strategy 1: Functional grouping
#     functional_groups = {
#         'living_entities': ['person', 'animal'],
#         'home_items': ['furniture', 'appliance', 'kitchen'],
#         'tech_items': ['electronic', 'vehicle'],
#         'consumables': ['food'],
#         'lifestyle': ['sports', 'accessory'],
#         'environments': ['indoor', 'outdoor']
#     }
    
#     # Strategy 2: Size-based grouping (merge smaller categories)
#     small_categories = [cat for cat, imgs in coco_data.items() if len(imgs) < 2000]
#     large_categories = [cat for cat, imgs in coco_data.items() if len(imgs) >= 2000]
    
#     size_based_groups = {
#         'miscellaneous': small_categories,
#         # Large categories remain separate
#     }
    
#     # Strategy 3: Semantic grouping
#     semantic_groups = {
#         'physical_objects': ['furniture', 'appliance', 'electronic', 'vehicle', 'accessory'],
#         'biological': ['person', 'animal'],
#         'edibles': ['food', 'kitchen'],
#         'activities': ['sports'],
#         'spaces': ['indoor', 'outdoor']
#     }
    
#     print("1. Functional grouping:")
#     for group, cats in functional_groups.items():
#         total_imgs = sum(len(coco_data[cat]) for cat in cats if cat in coco_data)
#         print(f"   {group}: {cats} → {total_imgs:,} images")
    
#     print("\n2. Size-based grouping (merge categories with <2000 images):")
#     print(f"   Small categories: {small_categories} → {sum(len(coco_data[cat]) for cat in small_categories):,} images")
#     print(f"   Large categories (kept separate): {large_categories}")
    
#     print("\n3. Semantic grouping:")
#     for group, cats in semantic_groups.items():
#         total_imgs = sum(len(coco_data[cat]) for cat in cats if cat in coco_data)
#         print(f"   {group}: {cats} → {total_imgs:,} images")
    
#     return {
#         'functional': functional_groups,
#         'size_based': size_based_groups,
#         'semantic': semantic_groups
#     }

# # Show regrouping options
# suggested_groups = create_custom_merge_groups()

# print("\n" + "="*60)
# print("To use any of these strategies, set merge_groups to one of:")
# print("  merge_groups = suggested_groups['functional']")
# print("  merge_groups = suggested_groups['size_based']") 
# print("  merge_groups = suggested_groups['semantic']")
# print("  merge_groups = {your_custom_groups}")
# print("="*60)
